# JFDS Experiment — 02: Main Experiments

**Part of a 5-notebook series** (see `README.md` for the full map):
`01_theory_and_hypothesis` → **`02_main_experiments`** → `03_ablation_and_complexity` →
`04_significance_and_robustness` → `05_appendix_supplementary`

This notebook is the core experimental pipeline: load MovieLens 1M, train SVD and BPR base
recommenders, build candidate pools, define the JFDS re-ranker plus the MMR and Fairness-only
baselines, run Bayesian optimisation (Optuna) to select λ_f / λ_d, generate recommendation lists
for all four strategies, and report the headline comparison table.

**Everything computed here is persisted to `jfds_artifacts.pkl` at the end** so that
`03`, `04`, and `05` can load it directly instead of re-running ~15+ minutes of training and
re-ranking. If you only want the core "does JFDS work" result, this notebook alone is sufficient.

# JFDS Experiment

In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit
from itertools import combinations
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity
import optuna
import numpy as np
import pandas as pd
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR           = Path("../dataset/movie_lens")
RANDOM_SEED        = 42
TEST_FRACTION      = 0.20
N_FACTORS          = 20
SVD_EPOCHS         = 12
BPR_EPOCHS         = 12
LEARNING_RATE      = 0.01
REG                = 0.02
BATCH_SIZE         = 8192
POOL_SIZE          = 50
K                  = 10
LAMBDA_F_DEFAULT   = 0.25
LAMBDA_D_DEFAULT   = 0.25
ANALYSIS_SAMPLE_N  = 1500

RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams['figure.dpi'] = 110
C_TOPK, C_JFDS, C_SVD, C_BPR = '#3498DB', '#E74C3C', '#9B59B6', '#16A085'
print('Config loaded.')


---
## 1. Load MovieLens 1M

In [ ]:
movies_df = pd.read_csv(DATA_DIR / 'movies.dat', sep='::', engine='python',
                        encoding='latin-1', header=None,
                        names=['MovieID', 'Title', 'Genres'])

users_df = pd.read_csv(DATA_DIR / 'users.dat', sep='::', engine='python',
                       encoding='latin-1', header=None,
                       names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip'])

ratings_df = pd.read_csv(DATA_DIR / 'ratings.dat', sep='::', engine='python',
                         encoding='latin-1', header=None,
                         names=['UserID', 'MovieID', 'Rating', 'Timestamp'])

print(f"movies : {len(movies_df):,}   users : {len(users_df):,}   ratings : {len(ratings_df):,}")
movies_df.head()


In [ ]:
# Contiguous ID remapping + genre vectors
unique_users  = np.sort(ratings_df['UserID'].unique())
unique_movies = np.sort(movies_df['MovieID'].unique())
user2idx  = {u: i for i, u in enumerate(unique_users)}
movie2idx = {m: i for i, m in enumerate(unique_movies)}
N_USERS, N_ITEMS = len(unique_users), len(unique_movies)

ratings_df['u_idx'] = ratings_df['UserID'].map(user2idx)
ratings_df['i_idx'] = ratings_df['MovieID'].map(movie2idx)
ratings_df = ratings_df.dropna(subset=['i_idx']).copy()
ratings_df['i_idx'] = ratings_df['i_idx'].astype(int)

movies_df = movies_df.set_index('MovieID').loc[unique_movies].reset_index()
all_genres = sorted({g for gl in movies_df['Genres'] for g in gl.split('|')})
N_GENRES = len(all_genres)
genre2idx = {g: i for i, g in enumerate(all_genres)}

genre_vec = np.zeros((N_ITEMS, N_GENRES))
for _, row in movies_df.iterrows():
    idx = movie2idx[row['MovieID']]
    for g in row['Genres'].split('|'):
        genre_vec[idx, genre2idx[g]] = 1.0
genre_vec = genre_vec / genre_vec.sum(axis=1, keepdims=True)

print("Pre-computing genre similarity matrix ...")
GENRE_SIM = cosine_similarity(genre_vec).astype(np.float32)
print(f"GENRE_SIM shape: {GENRE_SIM.shape}  dtype: {GENRE_SIM.dtype}")

users_df['u_idx'] = users_df['UserID'].map(user2idx)
users_df = users_df.dropna(subset=['u_idx']).copy()
users_df['u_idx'] = users_df['u_idx'].astype(int)

print(f"N_USERS={N_USERS}   N_ITEMS={N_ITEMS}   N_GENRES={N_GENRES}")
print('Genres:', all_genres)


## Data Preprocessing

This step prepares the dataset for model training by converting the original IDs into contiguous indices and generating genre based representations for movies.

### Steps

1. **Remap User and Movie IDs**
   - Convert original `UserID` and `MovieID` values into contiguous integer indices (`u_idx`, `i_idx`).
   - This makes embedding lookups efficient and compatible with deep learning frameworks.

2. **Filter and Align Data**
   - Remove ratings for movies that are not present in the movie metadata.
   - Reorder the movie dataset so that its rows match the new movie indices.

3. **Create Genre Vectors**
   - Extract all unique genres.
   - Represent each movie as a multi-hot vector indicating its genres.
   - Normalize each genre vector so its values sum to 1.

4. **Compute Genre Similarity**
   - Calculate a cosine similarity matrix between all movie genre vectors.
   - This matrix captures semantic similarity between movies based on shared genres and can be used for genre-aware recommendations or regularization.

5. **Remap User Metadata**
   - Apply the same user index mapping to the user metadata table to maintain consistency across all datasets.

### Output

- `u_idx`: Contiguous user indices.
- `i_idx`: Contiguous movie indices.
- `genre_vec`: Normalized genre representation for each movie.
- `GENRE_SIM`: Movie-to-movie genre similarity matrix.
- `N_USERS`, `N_ITEMS`, `N_GENRES`: Dataset statistics.

---
## 2. Train / Test Split

In [ ]:
ratings_df = ratings_df.sort_values(['UserID', 'Timestamp']).reset_index(drop=True)
ratings_df['rank_desc']  = ratings_df.groupby('UserID').cumcount(ascending=False)
ratings_df['user_count'] = ratings_df.groupby('UserID')['UserID'].transform('count')
test_cutoff = np.maximum(1, (ratings_df['user_count'] * TEST_FRACTION).astype(int))
test_mask = ratings_df['rank_desc'] < test_cutoff

train_df = ratings_df.loc[~test_mask, ['UserID','MovieID','Rating','Timestamp','u_idx','i_idx']].reset_index(drop=True)
test_df  = ratings_df.loc[ test_mask, ['UserID','MovieID','Rating','Timestamp','u_idx','i_idx']].reset_index(drop=True)

train_seen    = train_df.groupby('u_idx')['i_idx'].apply(set).to_dict()
test_relevant = test_df[test_df['Rating'] >= 4].groupby('u_idx')['i_idx'].apply(set).to_dict()
test_grades   = test_df.groupby('u_idx').apply(lambda g: dict(zip(g['i_idx'], g['Rating']))).to_dict()

print(f"train ratings: {len(train_df):,}   test ratings: {len(test_df):,}")
print(f"users with >=1 relevant test item: {len(test_relevant):,} / {N_USERS}")


In [ ]:
# Popularity tiers from TRAIN only
pop_count = np.zeros(N_ITEMS)
counts = train_df['i_idx'].value_counts()
pop_count[counts.index.values] = counts.values
pop_norm = pop_count / (pop_count.max() + 1e-12)

order_by_pop = np.argsort(-pop_count)
tier = np.empty(N_ITEMS, dtype=object)
tier[order_by_pop[:int(.2 * N_ITEMS)]]                  = 'head'
tier[order_by_pop[int(.2 * N_ITEMS):int(.5 * N_ITEMS)]] = 'mid'
tier[order_by_pop[int(.5 * N_ITEMS):]]                  = 'tail'
TIERS = ['head', 'mid', 'tail']
TARGET_SHARE = {t: 1/3 for t in TIERS}

print(pd.Series(tier).value_counts())


## Popularity Tier Assignment

Movies are grouped into popularity tiers using **only the training data**.

- Count interactions for each movie.
- Rank movies by popularity.
- Assign tiers:
  - **Head:** Top 20%
  - **Mid:** Next 30%
  - **Tail:** Remaining 50%
- Define an equal target recommendation share (33.3%) for each tier.

---
## 3. Base Recommenders: SVD and BPR 


In [ ]:
def train_svd(train_df, n_users, n_items, n_factors=N_FACTORS, n_epochs=SVD_EPOCHS,
              lr=LEARNING_RATE, reg=REG, batch_size=BATCH_SIZE, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    global_mean = train_df['Rating'].mean()
    b_u = np.zeros(n_users)
    b_i = np.zeros(n_items)
    P   = rng.normal(0, 0.1, (n_users, n_factors))
    Q   = rng.normal(0, 0.1, (n_items, n_factors))

    u_idx = train_df['u_idx'].values
    i_idx = train_df['i_idx'].values
    r_val = train_df['Rating'].values.astype(float)
    n = len(train_df)

    for epoch in range(n_epochs):
        perm = rng.permutation(n)
        for start in range(0, n, batch_size):
            b  = perm[start:start + batch_size]
            bu, bi, rv = u_idx[b], i_idx[b], r_val[b]

            pred = global_mean + b_u[bu] + b_i[bi] + np.einsum('bf,bf->b', P[bu], Q[bi])
            err  = rv - pred

            # ⚡ bincount scatter-add (replaces slow np.add.at)
            b_u += lr * (np.bincount(bu, weights=err - reg * b_u[bu], minlength=n_users))
            b_i += lr * (np.bincount(bi, weights=err - reg * b_i[bi], minlength=n_items))

            # Factor updates: accumulate per-index gradients then add once
            err_col = err[:, None]  # (B,1)
            P_grad = err_col * Q[bi] - reg * P[bu]   # (B, F)
            Q_grad = err_col * P[bu] - reg * Q[bi]   # (B, F)
            for f in range(n_factors):
                P[:, f] += lr * np.bincount(bu, weights=P_grad[:, f], minlength=n_users)
                Q[:, f] += lr * np.bincount(bi, weights=Q_grad[:, f], minlength=n_items)

        pred_all = global_mean + b_u[u_idx] + b_i[i_idx] + np.einsum('nf,nf->n', P[u_idx], Q[i_idx])
        rmse = np.sqrt(np.mean((r_val - pred_all) ** 2))
        print(f"  [SVD] epoch {epoch+1:>2}/{n_epochs}  train RMSE={rmse:.4f}")

    return {'global_mean': global_mean, 'b_u': b_u, 'b_i': b_i, 'P': P, 'Q': Q}

def svd_score_user(model, u):
    return model['global_mean'] + model['b_u'][u] + model['b_i'] + model['Q'] @ model['P'][u]


In [ ]:
print('Training SVD ...')
svd_model = train_svd(train_df, N_USERS, N_ITEMS)
print('SVD training complete.')


In [ ]:
def train_bpr(train_df, n_users, n_items, n_factors=N_FACTORS, n_epochs=BPR_EPOCHS,
              lr=LEARNING_RATE, reg=REG, batch_size=BATCH_SIZE, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    P   = rng.normal(0, 0.1, (n_users, n_factors))
    Q   = rng.normal(0, 0.1, (n_items, n_factors))
    b_i = np.zeros(n_items)

    pos_df    = train_df[train_df['Rating'] >= 4]
    pos_users = pos_df['u_idx'].values
    pos_items = pos_df['i_idx'].values
    n_pos = len(pos_users)

    for epoch in range(n_epochs):
        perm = rng.permutation(n_pos)
        for start in range(0, n_pos, batch_size):
            b  = perm[start:start + batch_size]
            bu, bi = pos_users[b], pos_items[b]
            bj = rng.integers(0, n_items, size=len(b))

            x_uij = np.einsum('bf,bf->b', P[bu], Q[bi] - Q[bj]) + b_i[bi] - b_i[bj]
            sig   = 1.0 / (1.0 + np.exp(np.clip(x_uij, -30, 30)))

            # ⚡ bincount scatter-add
            dQ = Q[bi] - Q[bj]

            for f in range(n_factors):
                P[:, f]  += lr * (np.bincount(bu, weights=sig * dQ[:, f],    minlength=n_users)
                                 - reg * np.bincount(bu, weights=P[bu, f], minlength=n_users))
                Q[:, f]  += lr * (np.bincount(bi, weights= sig * P[bu, f],   minlength=n_items)
                                 - reg * np.bincount(bi, weights=Q[bi, f], minlength=n_items))
                Q[:, f]  += lr * (np.bincount(bj, weights=-sig * P[bu, f],   minlength=n_items)
                                 - reg * np.bincount(bj, weights=Q[bj, f], minlength=n_items))

            b_i += lr * (np.bincount(bi, weights= sig - reg * b_i[bi], minlength=n_items)
                       + np.bincount(bj, weights=-sig - reg * b_i[bj], minlength=n_items))

        print(f"  [BPR] epoch {epoch+1:>2}/{n_epochs} complete")

    return {'P': P, 'Q': Q, 'b_i': b_i}

def bpr_score_user(model, u):
    return model['Q'] @ model['P'][u] + model['b_i']

In [ ]:
print('Training BPR ...')
bpr_model = train_bpr(train_df, N_USERS, N_ITEMS)
print('BPR training complete.')


In [ ]:
def build_score_matrix_svd(model):
    return (model['global_mean']
            + model['b_u'][:, None]
            + model['b_i'][None, :]
            + model['P'] @ model['Q'].T)   

def build_score_matrix_bpr(model):
    return model['P'] @ model['Q'].T + model['b_i'][None, :]

def build_candidates_fast(score_matrix, seen_dict, pool_size=POOL_SIZE):
    pools = {}
    for u in range(score_matrix.shape[0]):
        scores = score_matrix[u].copy()
        seen   = seen_dict.get(u, set())
        if seen:
            scores[list(seen)] = -np.inf
        top_idx = np.argpartition(scores, -pool_size)[-pool_size:]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        pools[u] = (top_idx, scores[top_idx])
    return pools

print('Building SVD score matrix ...')
svd_score_mat = build_score_matrix_svd(svd_model)
print('Building BPR score matrix ...')
bpr_score_mat = build_score_matrix_bpr(bpr_model)

print('Building candidate pools ...')
svd_pools = build_candidates_fast(svd_score_mat, train_seen)
bpr_pools = build_candidates_fast(bpr_score_mat, train_seen)
print(f"SVD pools: {len(svd_pools)}   BPR pools: {len(bpr_pools)}")


## Candidate Pool Generation

Candidate pools are created for both the SVD and BPR models.

- Compute recommendation scores for every user movie pair.
- Remove movies that users have already interacted with.
- Select the top scoring unseen movies as candidate recommendations.
- Store these candidate pools for efficient recommendation and evaluation.

---
## 4a. Rerankers

In [ ]:
# ──────────────────────────────────────────────────────────────
# Generic greedy reranking engine + Top-K baseline
# ──────────────────────────────────────────────────────────────
def greedy_rerank(cand_ids, cand_rel, score_fn, k=K, **params):
    rel_norm = (cand_rel - cand_rel.min()) / (cand_rel.max() - cand_rel.min() + 1e-12)
    rel_map  = dict(zip(cand_ids, rel_norm))
    remaining = list(cand_ids)
    selected, tier_counts = [], {t: 0 for t in TIERS}

    for step in range(min(k, len(cand_ids))):
        best_score, best_item = -np.inf, None
        for c in remaining:
            s = score_fn(c, rel_map[c], selected, tier_counts, step, **params)
            if s > best_score:
                best_score, best_item = s, c
        selected.append(best_item)
        tier_counts[tier[best_item]] += 1
        remaining.remove(best_item)
    return selected

def topk_rerank(cand_ids, cand_rel, k=K):
    order = np.argsort(-cand_rel)
    return list(np.array(cand_ids)[order[:k]])


### JFDS EQUATION

In [ ]:
def fair_boost(cand_idx, tier_counts, n_selected):
    t = tier[cand_idx]
    current_share = tier_counts[t] / max(1, n_selected)
    gap = max(0.0, TARGET_SHARE[t] - current_share)
    return (gap ** 2) / (TARGET_SHARE[t] ** 2)

def diversity_term_fast(cand_idx, selected_idx):
    """⚡ Uses pre-computed GENRE_SIM — no per-call cosine computation."""
    if not selected_idx:
        return 1.0
    return float(1.0 - GENRE_SIM[cand_idx, selected_idx].mean())

def jfds_score(cand_idx, rel_value, selected, tier_counts, step,
               lam_f=LAMBDA_F_DEFAULT, lam_d=LAMBDA_D_DEFAULT):
    lam_u = 1 - lam_f - lam_d
    return (lam_u * rel_value
            + lam_f * fair_boost(cand_idx, tier_counts, step)
            + lam_d * diversity_term_fast(cand_idx, selected))

def jfds_rerank(cand_ids, cand_rel, lam_f=LAMBDA_F_DEFAULT, lam_d=LAMBDA_D_DEFAULT, k=K):
    return greedy_rerank(cand_ids, cand_rel, jfds_score, k=k, lam_f=lam_f, lam_d=lam_d)


---
### Established Baselines: MMR and Fairness-Only

To make the case for JFDS credible we compare it against two established,
**single-objective** re-ranking methods that each isolate one half of what
JFDS does jointly. Both reuse the same `greedy_rerank` engine and the same
`GENRE_SIM` / `tier` data structures as JFDS, so every strategy sees an
identical candidate pool and an identical notion of "diversity" and
"fairness" — a fair head-to-head comparison, not an apples-to-oranges one.

- **MMR (Carbonell & Goldstein, 1998)** — diversity only, no fairness term:
  $$\text{score}(i,S) = \lambda \cdot rel(i) - (1-\lambda)\max_{j \in S} sim(i,j)$$
- **Fairness-only** — exposure-fairness only, no diversity term:
  $$\text{score}(i) = (1-\lambda_f)\cdot rel(i) + \lambda_f \cdot fair\_boost(i)$$

If JFDS outperforms **both** specialists on the metric each was designed to
win — without giving up materially more relevance — that is much stronger
evidence than beating a no-objective baseline (Top-K) alone.


In [ ]:
MMR_LAMBDA        = 0.7    # Strategy C (MMR) — fixed relevance weight, standard literature value
FAIR_ONLY_LAMBDA  = 0.45   # Strategy D (fairness-only) — fixed fairness weight, matches JFDS's optimizable upper range

# ── Strategy C: MMR — diversity only, no fairness term ──────────────────────
def mmr_max_sim(cand_idx, selected_idx):
    """Max genre-cosine similarity between a candidate and everything already selected."""
    if not selected_idx:
        return 0.0
    return float(GENRE_SIM[cand_idx, selected_idx].max())

def mmr_score(cand_idx, rel_value, selected, tier_counts, step, lam=MMR_LAMBDA):
    return lam * rel_value - (1 - lam) * mmr_max_sim(cand_idx, selected)

def mmr_rerank(cand_ids, cand_rel, lam=MMR_LAMBDA, k=K):
    return greedy_rerank(cand_ids, cand_rel, mmr_score, k=k, lam=lam)

# ── Strategy D: Fairness-only re-ranking, no diversity term ──────────────
def fairness_only_score(cand_idx, rel_value, selected, tier_counts, step, lam_f=FAIR_ONLY_LAMBDA):
    return (1 - lam_f) * rel_value + lam_f * fair_boost(cand_idx, tier_counts, step)

def fairness_only_rerank(cand_ids, cand_rel, lam_f=FAIR_ONLY_LAMBDA, k=K):
    return greedy_rerank(cand_ids, cand_rel, fairness_only_score, k=k, lam_f=lam_f)

print('Strategy C (MMR, diversity-only) defined ✓  —  score = λ·rel − (1−λ)·max_sim,  λ =', MMR_LAMBDA)
print('Strategy D (fairness-only) defined ✓        —  score = (1−λ_f)·rel + λ_f·fair_boost,  λ_f =', FAIR_ONLY_LAMBDA)


---
## 4b. Lambda Grid Search (Bayesian Optimisation via Optuna)

In [ ]:
# Lambda grid-search hyper-params
LAMBDA_STEP  = 0.10
VAL_FRACTION = 0.10
BETA_F       = 0.5
BETA_D       = 0.5
# MMR_LAMBDA / FAIR_ONLY_LAMBDA are defined earlier, in the baseline-reranker cell (Section 4a), since they're needed there first.
print('Lambda grid-search config loaded.')


### Core Metric Helpers

`ild`, `precision_recall_ndcg`, and `novelty_fairness` are defined once, here,
because they're needed both inside the Optuna objective below (Section 4b)
and later throughout evaluation (Section 5 onward) — defining them twice was
redundant and risked the two copies drifting apart.


In [ ]:
# ── Core metric helpers (single canonical definition, reused everywhere) ─────
def ild(rec_list):
    """Intra-list diversity: 1 - mean pairwise genre-cosine similarity."""
    if len(rec_list) < 2:
        return 0.0
    idx = np.array(rec_list)
    sub = GENRE_SIM[np.ix_(idx, idx)]   # ⚡ vectorised: submatrix of pre-computed GENRE_SIM
    n = len(idx)
    mask = np.triu(np.ones((n, n), dtype=bool), k=1)
    return float(1.0 - sub[mask].mean())

def precision_recall_ndcg(rec_list, relevant_set, grades, k=K):
    rec   = rec_list[:k]
    hits  = len(set(rec) & relevant_set)
    prec  = hits / k
    rec_r = hits / max(1, len(relevant_set))
    dcg   = sum(grades.get(item, 0) / np.log2(r + 2) for r, item in enumerate(rec))
    idcg  = sum(g / np.log2(r + 2) for r, g in enumerate(sorted(grades.values(), reverse=True)[:k]))
    return prec, rec_r, (dcg / idcg if idcg > 0 else 0.0)

def novelty_fairness(rec_list):
    """Mean (1 - normalised popularity): higher = more novel/niche items recommended."""
    return float(np.mean([1 - pop_norm[i] for i in rec_list]))

print('Core metric helpers (ild, precision_recall_ndcg, novelty_fairness) defined ✓')


In [ ]:
# Step 1: carve validation split from training data
train_df_sorted = train_df.sort_values(['UserID','Timestamp']).reset_index(drop=True)
train_df_sorted['rank_desc_tr'] = train_df_sorted.groupby('UserID').cumcount(ascending=False)
train_df_sorted['user_count_tr'] = train_df_sorted.groupby('UserID')['UserID'].transform('count')
val_cutoff   = np.maximum(1, (train_df_sorted['user_count_tr'] * VAL_FRACTION).astype(int))
val_mask_tr  = train_df_sorted['rank_desc_tr'] < val_cutoff

val_tr_df   = train_df_sorted.loc[ val_mask_tr].copy()
pure_tr_df  = train_df_sorted.loc[~val_mask_tr].copy()

val_relevant_tr = val_tr_df[val_tr_df['Rating'] >= 4].groupby('u_idx')['i_idx'].apply(set).to_dict()
val_grades_tr   = val_tr_df.groupby('u_idx').apply(lambda g: dict(zip(g['i_idx'], g['Rating']))).to_dict()
pure_seen_tr    = pure_tr_df.groupby('u_idx')['i_idx'].apply(set).to_dict()

# Validation pools also use the fast matrix approach
def build_val_score_matrix_svd(model, pure_seen_dict):
    """Same matrix, but mask pure_seen items per user before argpartition."""
    mat = build_score_matrix_svd(model).copy()
    for u, seen in pure_seen_dict.items():
        mat[u, list(seen)] = -np.inf
    return mat

def build_val_score_matrix_bpr(model, pure_seen_dict):
    mat = build_score_matrix_bpr(model).copy()
    for u, seen in pure_seen_dict.items():
        mat[u, list(seen)] = -np.inf
    return mat

print('Building validation candidate pools ...')
svd_val_mat  = build_val_score_matrix_svd(svd_model, pure_seen_tr)
bpr_val_mat  = build_val_score_matrix_bpr(bpr_model, pure_seen_tr)
svd_val_pools = build_candidates_fast(svd_val_mat, {})   # seen already masked in matrix
bpr_val_pools = build_candidates_fast(bpr_val_mat, {})
print(f'  SVD val pools: {len(svd_val_pools)}   BPR val pools: {len(bpr_val_pools)}')

# Step 2: diversity target = mean ILD of Top-K on val (uses the canonical `ild` defined above)
def mean_ild_topk(val_pools):
    return float(np.mean([ild(topk_rerank(cids, crel)) for _, (cids, crel) in val_pools.items()]))

TARGET_D_SVD = mean_ild_topk(svd_val_pools)
TARGET_D_BPR = mean_ild_topk(bpr_val_pools)
print(f'Diversity targets — SVD: {TARGET_D_SVD:.4f}   BPR: {TARGET_D_BPR:.4f}')


In [ ]:


# Composite objective for the Optuna search (uses the canonical metric helpers defined above)
def composite_score_user(rec, relevant_set, grades, target_d):
    if not rec:
        return -np.inf
    _, _, ndcg = precision_recall_ndcg(rec, relevant_set, grades)
    d_val = ild(rec)
    f_val = novelty_fairness(rec)
    return ndcg - BETA_F * abs(f_val - 1/3) - BETA_D * abs(d_val - target_d)

def optimize_lambdas(val_pools, target_d, name, n_trials=50):
    users = [(u, cids, crel) for u, (cids, crel) in val_pools.items()]
    history = []

    def objective(trial):
        # Restrict search to a clean 2-decimal-place grid: 0.00, 0.01, ..., 1.00
        lam_f = round(trial.suggest_float("lam_f", 0.0, 1.0, step=0.01), 2)

        hi = round(1.0 - lam_f, 2)
        if hi <= 0:
            lam_d = 0.0
            trial.set_user_attr("lam_d_forced", True)
        else:
            lam_d = round(trial.suggest_float("lam_d", 0.0, hi, step=0.01), 2)

        scores = []
        for u, cids, crel in users:
            rel = val_relevant_tr.get(u, set())
            grd = val_grades_tr.get(u, {})
            rec = jfds_rerank(cids, crel, lam_f=lam_f, lam_d=lam_d)
            scores.append(composite_score_user(rec, rel, grd, target_d))

        mean_score = float(np.mean(scores))
        history.append({'lam_f': lam_f, 'lam_d': lam_d, 'composite': mean_score})
        return mean_score

    study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_lf = round(study.best_params["lam_f"], 2)
    best_ld = round(study.best_params.get("lam_d", 0.0), 2)
    best_lu = round(1 - best_lf - best_ld, 2)

    print(f"[{name}] best λ_f={best_lf:.2f} λ_d={best_ld:.2f} "
          f"λ_u={best_lu:.2f} composite={study.best_value:.5f}")

    return best_lf, best_ld, pd.DataFrame(history)

print("Running Bayesian Optimisation ...")
best_lf_svd, best_ld_svd, optuna_svd = optimize_lambdas(svd_val_pools, TARGET_D_SVD, "SVD", n_trials=50)
best_lf_bpr, best_ld_bpr, optuna_bpr = optimize_lambdas(bpr_val_pools, TARGET_D_BPR, "BPR", n_trials=50)

best_lambdas = {
    "SVD": {"lam_f": best_lf_svd, "lam_d": best_ld_svd},
    "BPR": {"lam_f": best_lf_bpr, "lam_d": best_ld_bpr},
}
print("\nBest lambdas:", best_lambdas)

## Hyperparameter Optimization

The optimal fairness (`λ_f`) and diversity (`λ_d`) weights are selected using a validation set and Bayesian Optimization.

- Split the training data into training and validation subsets.
- Generate validation candidate pools for the SVD and BPR models.
- Compute a target diversity score from the validation recommendations.
- Evaluate recommendations using a composite objective that balances ranking quality (NDCG), fairness, and diversity.
- Use Bayesian Optimization (Optuna) to find the best values of `λ_f` and `λ_d` for each model.

In [ ]:
# Visualise Optuna trial history as scatter (lam_f vs lam_d, colour = composite)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (name, hist_df), best_lf, best_ld in zip(
        axes,
        [('SVD', optuna_svd), ('BPR', optuna_bpr)],
        [best_lf_svd, best_lf_bpr],
        [best_ld_svd, best_ld_bpr]):
    sc = ax.scatter(hist_df['lam_f'], hist_df['lam_d'],
                    c=hist_df['composite'], cmap='RdYlGn', s=60, edgecolor='k', linewidth=0.4)
    ax.scatter([best_lf], [best_ld], marker='*', s=280, color='white',
               edgecolor='black', linewidth=1.2, zorder=5,
               label=f'best: λ_f={best_lf:.2f}, λ_d={best_ld:.2f}')
    ax.set_xlabel('λ_f (fairness weight)')
    ax.set_ylabel('λ_d (diversity weight)')
    ax.set_title(f'{name}: Optuna trial history\n(white star = selected λ)', fontsize=11)
    ax.legend(fontsize=9)
    plt.colorbar(sc, ax=ax, label='composite score')

plt.suptitle('Lambda Search — Bayesian Optimisation', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('lambda_grid_search.png', dpi=110, bbox_inches='tight')
plt.show()


In [ ]:
# Re-rank all users using best lambdas (JFDS) plus the two fixed-weight baselines (MMR, Fairness-only)
print('Re-ranking all users (Top-K, JFDS, MMR, Fairness-only) ...')
lists = {}
for model_name, pools in [('SVD', svd_pools), ('BPR', bpr_pools)]:
    lf = best_lambdas[model_name]['lam_f']
    ld = best_lambdas[model_name]['lam_d']
    topk_lists, jfds_lists, mmr_lists, fair_lists = {}, {}, {}, {}
    for u in range(N_USERS):
        cand_ids, cand_rel = pools[u]
        if len(cand_ids) == 0:
            continue
        topk_lists[u] = topk_rerank(cand_ids, cand_rel)
        jfds_lists[u] = jfds_rerank(cand_ids, cand_rel, lam_f=lf, lam_d=ld)
        mmr_lists[u]  = mmr_rerank(cand_ids, cand_rel)
        fair_lists[u] = fairness_only_rerank(cand_ids, cand_rel)
    lists[(model_name, 'TopK')]    = topk_lists
    lists[(model_name, 'JFDS')]    = jfds_lists
    lists[(model_name, 'MMR')]     = mmr_lists
    lists[(model_name, 'FairOnly')] = fair_lists
    print(f"  {model_name}: λ_f={lf:.2f} λ_d={ld:.2f}  |  MMR λ={MMR_LAMBDA}  |  FairOnly λ_f={FAIR_ONLY_LAMBDA}  — "
          f"{len(topk_lists)} Top-K, {len(jfds_lists)} JFDS, {len(mmr_lists)} MMR, {len(fair_lists)} FairOnly lists")


---
## Sanity Check: One User, Four Strategies Side by Side

Before trusting the aggregate metrics below, spot-check a single user: print
the Top-10 list each strategy actually produces for them (SVD candidates),
so the numbers in the rest of the notebook can be tied back to real,
readable recommendations.


In [ ]:
# ── Single-user sanity check (SVD candidates) ──────────────────────
SAMPLE_U = next(iter(lists[('SVD', 'JFDS')].keys()))
idx2movie = {v: k for k, v in movie2idx.items()}

sample_rows = []
for method in ['TopK', 'JFDS', 'MMR', 'FairOnly']:
    rec = lists[('SVD', method)].get(SAMPLE_U, [])
    titles = movies_df.set_index('MovieID').loc[[idx2movie[i] for i in rec], 'Title'].tolist()
    for rank, (item_idx, title) in enumerate(zip(rec, titles), start=1):
        sample_rows.append({'Strategy': method, 'Rank': rank, 'Title': title, 'Tier': tier[item_idx]})

sample_df = pd.DataFrame(sample_rows)
print(f"Sample user (SVD, u_idx={SAMPLE_U}) — already rated {len(train_seen.get(SAMPLE_U, set()))} movies in training")
for method in ['TopK', 'JFDS', 'MMR', 'FairOnly']:
    print(f"\n=== {method} ===")
    print(sample_df[sample_df.Strategy == method][['Rank', 'Title', 'Tier']].to_string(index=False))

ids_topk = set(lists[('SVD', 'TopK')].get(SAMPLE_U, []))
ids_jfds = set(lists[('SVD', 'JFDS')].get(SAMPLE_U, []))
overlap  = ids_topk & ids_jfds
print(f"\nTop-K vs JFDS overlap for this user: {len(overlap)} / {len(ids_topk)} items in common")


---
## 5. Evaluation Metrics 

In [ ]:
# `precision_recall_ndcg`, `ild`, and `novelty_fairness` are already defined above (Section 4b) and reused here unchanged.
def shannon_entropy(rec_list):
    dist = genre_vec[rec_list].mean(axis=0)
    dist = dist / dist.sum()
    return float(-np.sum([p * np.log2(p) for p in dist if p > 0]))

def gini(values):
    v = np.sort(np.asarray(values, dtype=float))
    n = len(v)
    if v.sum() == 0:
        return 0.0
    cum = np.cumsum(v)
    return float((n + 1 - 2 * np.sum(cum) / cum[-1]) / n)

def calibration_kl(rec_list, user_train_items, alpha=0.01):
    if len(user_train_items) == 0:
        return 0.0
    p = genre_vec[list(user_train_items)].mean(axis=0)
    p = p / p.sum()
    q = genre_vec[rec_list].mean(axis=0)
    q = (1 - alpha) * q + alpha * p
    q = q / q.sum()
    return float(np.sum(p * np.log((p + 1e-12) / (q + 1e-12))))

# ── Aggregate diversity helpers (for R10) ─────────────────────────────────────
def aggregate_diversity(rec_lists_dict):
    all_items = set()
    for rec in rec_lists_dict.values():
        all_items.update(rec)
    return len(all_items)

def exposure_vector(rec_lists_dict, n_items=N_ITEMS):
    exp = np.zeros(n_items)
    for rec in rec_lists_dict.values():
        for i in rec:
            exp[i] += 1
    return exp


In [ ]:
# Per-user metrics for every (base_model, method) combination
rows = []
for (model_name, method), rec_lists in lists.items():
    for u, rec in rec_lists.items():
        relevant_set = test_relevant.get(u, set())
        grades       = test_grades.get(u, {})
        p, r, n_ndcg = precision_recall_ndcg(rec, relevant_set, grades)
        rows.append({
            'user': u, 'base_model': model_name, 'method': method,
            'precision': p, 'recall': r, 'ndcg': n_ndcg,
            'D': ild(rec), 'F': novelty_fairness(rec),
            'H': shannon_entropy(rec), 'gini_exp': gini(pop_count[rec]),
            'calibration_kl': calibration_kl(rec, train_seen.get(u, set())),
        })

metrics_long = pd.DataFrame(rows)
metrics_long['U'] = metrics_long['ndcg']
print(f"metrics_long: {len(metrics_long):,} rows")
metrics_long.head()


In [ ]:
# Headline comparison table
summary = (metrics_long
           .groupby(['base_model', 'method'])
           [['precision', 'recall', 'ndcg', 'D', 'F', 'H', 'gini_exp', 'calibration_kl']]
           .mean().round(4))

sys_rows = []
for (model_name, method), rec_lists in lists.items():
    exposure   = exposure_vector(rec_lists)
    coverage   = (exposure > 0).sum() / N_ITEMS
    exp_gini   = gini(exposure)
    agg_div    = aggregate_diversity(rec_lists)
    tier_share = (pd.Series(tier[np.repeat(np.arange(N_ITEMS), exposure.astype(int))])
                  .value_counts(normalize=True).reindex(TIERS).fillna(0))
    sys_rows.append({'base_model': model_name, 'method': method,
                     'agg_diversity': agg_div, 'coverage': coverage,
                     'exposure_gini': exp_gini,
                     'head_share': tier_share['head'],
                     'mid_share':  tier_share['mid'],
                     'tail_share': tier_share['tail']})

system_summary = pd.DataFrame(sys_rows).set_index(['base_model', 'method']).round(4)
display(summary)
display(system_summary)


---
## Baseline Comparison — JFDS vs Top-K vs MMR vs Fairness-only

The headline table above already contains all four methods (Top-K, JFDS,
MMR, Fairness-only) for both base recommenders, because they were folded
into the same `lists` / `metrics_long` structures used everywhere else in
this notebook. This section just visualizes that comparison directly:

1. Bar charts of Fairness (F), Diversity (D), and NDCG for all four
   strategies, per base model.
2. A delta table quantifying exactly what JFDS gains or gives up relative
   to each specialist baseline.


In [ ]:
# ── Bar chart: F / D / NDCG across all 4 strategies, per base model ────────
method_order = ['TopK', 'MMR', 'FairOnly', 'JFDS']
method_colors = {'TopK': '#7F8C8D', 'MMR': '#2980B9', 'FairOnly': '#E67E22', 'JFDS': '#27AE60'}
metrics_bar = ['F', 'D', 'ndcg']
metric_titles = {'F': 'Fairness (novelty)', 'D': 'Diversity (ILD)', 'ndcg': 'NDCG@10 (utility)'}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for row, model_name in enumerate(['SVD', 'BPR']):
    for col, metric in enumerate(metrics_bar):
        ax = axes[row][col]
        vals = [metrics_long[(metrics_long.base_model == model_name) &
                              (metrics_long.method == m)][metric].mean() for m in method_order]
        bars = ax.bar(method_order, vals, color=[method_colors[m] for m in method_order], edgecolor='black')
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width()/2, v, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
        ax.set_title(f'{model_name}: {metric_titles[metric]}', fontsize=10)
        ax.tick_params(axis='x', rotation=20)

plt.suptitle('JFDS vs. Top-K, MMR (diversity-only), and Fairness-only baselines', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('baseline_comparison_bars.png', dpi=110, bbox_inches='tight')
plt.show()

# ── Delta table: JFDS vs each established baseline, per base model ──────────
delta_rows_base = []
for model_name in ['SVD', 'BPR']:
    jfds_means = metrics_long[(metrics_long.base_model == model_name) &
                               (metrics_long.method == 'JFDS')][metrics_bar].mean()
    for base_method in ['TopK', 'MMR', 'FairOnly']:
        base_means = metrics_long[(metrics_long.base_model == model_name) &
                                   (metrics_long.method == base_method)][metrics_bar].mean()
        for metric in metrics_bar:
            b_val, base_val = jfds_means[metric], base_means[metric]
            pct = (b_val - base_val) / base_val * 100 if base_val != 0 else np.nan
            delta_rows_base.append({
                'base_model': model_name, 'baseline': base_method, 'metric': metric,
                'JFDS': round(b_val, 4), 'baseline_value': round(base_val, 4),
                'delta': round(b_val - base_val, 4), 'pct_change': round(pct, 2),
            })

baseline_delta_df = pd.DataFrame(delta_rows_base)
display(baseline_delta_df)


---
## Persisting Artifacts for Downstream Notebooks

Everything below is unchanged computation — just packaging what's already in memory into a
single pickle file so `03_ablation_and_complexity`, `04_significance_and_robustness`, and
`05_appendix_supplementary` can load it without re-running SVD/BPR training or the Optuna search.

The file will be a few hundred MB (dominated by `GENRE_SIM`, the candidate pools, and the
per-strategy recommendation lists for ~6,000 users) — this is expected and fine for local/Colab use.

In [ ]:
import pickle

ARTIFACTS = {
    # Dataset-scale constants
    'N_USERS': N_USERS, 'N_ITEMS': N_ITEMS, 'N_GENRES': N_GENRES,
    'N_FACTORS': N_FACTORS, 'K': K, 'POOL_SIZE': POOL_SIZE,
    'RANDOM_SEED': RANDOM_SEED, 'ANALYSIS_SAMPLE_N': ANALYSIS_SAMPLE_N,
    'TEST_FRACTION': TEST_FRACTION,

    # Tiers / fairness targets
    'TIERS': TIERS, 'TARGET_SHARE': TARGET_SHARE, 'tier': tier,

    # Item / genre representations
    'genre_vec': genre_vec, 'GENRE_SIM': GENRE_SIM,
    'pop_count': pop_count, 'pop_norm': pop_norm,
    'all_genres': all_genres, 'N_GENRES': N_GENRES,

    # Raw + mapping tables
    'movies_df': movies_df, 'users_df': users_df,
    'movie2idx': movie2idx, 'user2idx': user2idx,

    # Train/test structures
    'train_seen': train_seen, 'test_relevant': test_relevant, 'test_grades': test_grades,

    # Trained base recommenders + candidate pools
    'svd_model': svd_model, 'bpr_model': bpr_model,
    'svd_pools': svd_pools, 'bpr_pools': bpr_pools,

    # Selected hyperparameters
    'best_lambdas': best_lambdas, 'MMR_LAMBDA': MMR_LAMBDA, 'FAIR_ONLY_LAMBDA': FAIR_ONLY_LAMBDA,

    # Results
    'lists': lists, 'metrics_long': metrics_long,
    'summary': summary, 'system_summary': system_summary,
    'baseline_delta_df': baseline_delta_df,

    # Plot colours (kept consistent across all notebooks)
    'C_TOPK': C_TOPK, 'C_JFDS': C_JFDS, 'C_SVD': C_SVD, 'C_BPR': C_BPR,
}

with open('jfds_artifacts.pkl', 'wb') as f:
    pickle.dump(ARTIFACTS, f, protocol=pickle.HIGHEST_PROTOCOL)

import os
size_mb = os.path.getsize('jfds_artifacts.pkl') / 1e6
print(f"Saved {len(ARTIFACTS)} objects to jfds_artifacts.pkl  ({size_mb:.1f} MB)")
print("Load in downstream notebooks with:")
print("    import pickle; ARTIFACTS = pickle.load(open('jfds_artifacts.pkl','rb')); globals().update(ARTIFACTS)")
